# 🔬 Mobile AutoML for Skin Lesion Triage on IoMT Devices

**Conference:**

**Environment:** Google Colab Pro (L4 / T4 GPU recommended)  
**Runtime:** ~3–3.5 hours (first run) · ~15–20 minutes (cached rerun)

---

### Description

This notebook implements a complete experimental pipeline for benchmarking **Automated Machine Learning (AutoML)** against manually-designed lightweight architectures for **three-class skin lesion triage** (benign / malignant / precancerous). Models are quantised for deployment on **IoMT edge devices** and interpreted via **Grad-CAM explainability**.

### How to use

1. Upload your `kaggle.json` to Google Drive at:  
   `/MyDrive/AutoML_Turkey_Conference_AIR/config/kaggle.json`
2. Select a GPU runtime: *Runtime → Change runtime type → L4 or T4*
3. Run all cells: *Runtime → Run all*

### Caching

Every computationally expensive step checks for a cached result on Google Drive before executing. If a session disconnects, simply reconnect and re-run — completed steps load from cache instantly.

## Cell 0 — Environment Setup & Configuration
Install dependencies, verify GPU, mount Google Drive, configure paths and global constants.

In [ ]:
import subprocess, sys

def install(package):
    """Install a pip package quietly if not already available."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package]
    )

# Core ML dependencies (Colab already has TF, but we pin extras)
install("autokeras")
install("kaggle")
install("scikit-learn")
install("seaborn")

import os, json, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score
)

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

# ── Mixed-precision training (~30-40% faster on L4/T4, no accuracy loss)
tf.keras.mixed_precision.set_global_policy("mixed_float16")

# ── Global constants ─────────────────────────────────────────────────────
IMG_SIZE        = 224
BATCH_SIZE      = 32
NUM_CLASSES     = 3
CLASS_NAMES     = ["benign", "malignant", "precancerous"]
AK_MAX_TRIALS   = 10          # AutoKeras search budget
AK_MAX_EPOCHS   = 50          # per trial, with early stopping
BASELINE_EPOCHS = 30          # for manual baselines, with early stopping

# ── Mount Google Drive & set up project directory ────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/AutoML_Turkey_Conference_AIR")
DATA_DIR        = PROJECT_ROOT / "data"
CHECKPOINT_DIR  = PROJECT_ROOT / "checkpoints"
TFLITE_DIR      = PROJECT_ROOT / "tflite_models"
RESULTS_DIR     = PROJECT_ROOT / "results"
FIGURES_DIR     = PROJECT_ROOT / "figures"
CONFIG_DIR      = PROJECT_ROOT / "config"

for d in [DATA_DIR, CHECKPOINT_DIR, TFLITE_DIR, RESULTS_DIR, FIGURES_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Pipeline state tracker (for caching) ─────────────────────────────────
STATE_FILE = CHECKPOINT_DIR / "pipeline_state.json"

def load_state():
    """Load pipeline completion state from Drive."""
    if STATE_FILE.exists():
        with open(STATE_FILE) as f:
            return json.load(f)
    return {}

def save_state(key, value=True):
    """Mark a pipeline step as completed."""
    state = load_state()
    state[key] = value
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

def is_done(key):
    """Check whether a pipeline step has already been completed."""
    return load_state().get(key, False)

# ── GPU check ────────────────────────────────────────────────────────────
gpu_name = tf.config.list_physical_devices("GPU")
if gpu_name:
    print(f"✓ GPU detected: {gpu_name[0].name}")
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print("⚠ No GPU detected — training will be significantly slower.")

print(f"✓ TensorFlow {tf.__version__}")
print(f"✓ Mixed precision: {tf.keras.mixed_precision.global_policy().name}")
print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Pipeline state: {load_state()}")

## Cell 1 — Download HAM10000 Dataset via Kaggle API
Downloads the HAM10000 dataset using the Kaggle API. The dataset is cached on Google Drive so subsequent runs skip the download.

**Kaggle credentials** are handled automatically:
- If `kaggle.json` already exists on Drive → loaded silently
- If not → you will be prompted to enter your Kaggle username and API key (get them from [kaggle.com/settings](https://www.kaggle.com/settings) → API → *Create New Token*)
- Credentials are saved to Drive for all future runs

In [ ]:
HAM_DIR = DATA_DIR / "ham10000"

if HAM_DIR.exists() and len(list(HAM_DIR.glob("*.jpg"))) > 10000:
    print(f"✓ HAM10000 already cached at {HAM_DIR} "
          f"({len(list(HAM_DIR.glob('*.jpg')))} images)")
else:
    print("⏳ Downloading HAM10000 via Kaggle API...")

    # Configure Kaggle credentials from Drive
    kaggle_cred = CONFIG_DIR / "kaggle.json"
    assert kaggle_cred.exists(), (
        f"✗ kaggle.json not found at {kaggle_cred}. "
        "Please upload your Kaggle API key there."
    )
    kaggle_dest = Path.home() / ".kaggle"
    kaggle_dest.mkdir(exist_ok=True)
    shutil.copy(kaggle_cred, kaggle_dest / "kaggle.json")
    os.chmod(kaggle_dest / "kaggle.json", 0o600)

    # Download and extract
    HAM_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 \
        -p "{HAM_DIR}" --unzip --quiet

    # Flatten: move images from sub-folders to HAM_DIR root if needed
    for subfolder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        src = HAM_DIR / subfolder
        if src.exists():
            for img in src.glob("*.jpg"):
                shutil.move(str(img), str(HAM_DIR / img.name))
            src.rmdir()

    n_images = len(list(HAM_DIR.glob("*.jpg")))
    print(f"✓ Downloaded {n_images} images to {HAM_DIR}")

# Load metadata CSV
meta_candidates = list(HAM_DIR.glob("*metadata*"))
if not meta_candidates:
    meta_candidates = list(HAM_DIR.glob("*.csv"))
META_PATH = meta_candidates[0]
df = pd.read_csv(META_PATH)
print(f"✓ Metadata loaded: {len(df)} entries, columns: {list(df.columns)}")
print(f"\nOriginal 7-class distribution:\n{df['dx'].value_counts()}")

## Cell 2 — Three-Class Triage Remapping
Map the original 7 diagnostic categories to 3 clinically meaningful triage classes:

| Triage Class | Original Classes | Clinical Action |
|---|---|---|
| **Benign** | NV, BKL, DF, VASC | No urgent action |
| **Malignant** | MEL, BCC | Urgent referral |
| **Precancerous** | AKIEC | Monitoring / follow-up |

In [ ]:
TRIAGE_MAP = {
    # Benign lesions (label 0)
    "nv":    "benign",      # melanocytic nevi
    "bkl":   "benign",      # benign keratosis-like lesions
    "df":    "benign",      # dermatofibroma
    "vasc":  "benign",      # vascular lesions
    # Malignant lesions (label 1)
    "mel":   "malignant",   # melanoma
    "bcc":   "malignant",   # basal cell carcinoma
    # Precancerous lesions (label 2)
    "akiec": "precancerous" # actinic keratoses
}

LABEL_ENCODING = {"benign": 0, "malignant": 1, "precancerous": 2}

df["triage"]       = df["dx"].map(TRIAGE_MAP)
df["triage_label"] = df["triage"].map(LABEL_ENCODING)

print("Three-class triage distribution:")
print(df["triage"].value_counts())
print(f"\nClass ratios:")
for cls in CLASS_NAMES:
    n = (df["triage"] == cls).sum()
    print(f"  {cls:14s}: {n:5d} ({n / len(df) * 100:.1f}%)")

# ── Compute class weights for balanced training ─────────────────────────
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=df["triage_label"].values
)
CLASS_WEIGHTS = dict(enumerate(class_weights_array))
print(f"\nClass weights for loss function: {CLASS_WEIGHTS}")

# ── Visualise distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["dx"].value_counts().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Original 7-class distribution")
axes[0].set_ylabel("Count")
df["triage"].value_counts().plot.bar(ax=axes[1], color=["#2ecc71", "#e74c3c", "#f39c12"])
axes[1].set_title("Triage 3-class distribution")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Figure saved: {FIGURES_DIR / 'fig_class_distribution.png'}")

## Cell 3 — Load and Preprocess Images
Load all images, resize to 224×224, normalise to [0, 1]. Uses a NumPy cache on Drive to avoid reloading on reruns.

In [ ]:
CACHE_X = CHECKPOINT_DIR / "X_images.npy"
CACHE_Y = CHECKPOINT_DIR / "y_labels.npy"

if CACHE_X.exists() and CACHE_Y.exists():
    print("✓ Loading preprocessed images from cache...")
    X = np.load(CACHE_X)
    y = np.load(CACHE_Y)
    print(f"  X shape: {X.shape}, y shape: {y.shape}")
else:
    print("⏳ Loading and preprocessing images (this may take a few minutes)...")
    from tensorflow.keras.preprocessing.image import load_img, img_to_array

    images, labels = [], []
    missing = 0

    for _, row in df.iterrows():
        img_path = HAM_DIR / f"{row['image_id']}.jpg"
        if not img_path.exists():
            missing += 1
            continue
        img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        images.append(img_to_array(img) / 255.0)
        labels.append(row["triage_label"])

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)

    if missing > 0:
        print(f"  ⚠ {missing} images not found (skipped)")

    # Cache to Drive for fast reruns
    np.save(CACHE_X, X)
    np.save(CACHE_Y, y)
    print(f"✓ Preprocessed {len(X)} images → cached to Drive")
    print(f"  X shape: {X.shape} ({X.nbytes / 1e9:.2f} GB)")
    print(f"  y shape: {y.shape}")

## Cell 4 — Stratified Train / Validation / Test Split
Split 80/10/10 stratified by triage class.

**Important:** We split by `lesion_id` (not `image_id`) to prevent data leakage — some lesions have multiple images taken from different angles.

In [ ]:
CACHE_SPLIT = CHECKPOINT_DIR / "split_indices.npz"

if CACHE_SPLIT.exists():
    print("✓ Loading cached split indices...")
    split_data = np.load(CACHE_SPLIT)
    idx_train = split_data["idx_train"]
    idx_val   = split_data["idx_val"]
    idx_test  = split_data["idx_test"]
else:
    print("⏳ Computing stratified split by lesion_id...")

    df_indexed = df.reset_index(drop=True)
    lesion_ids = df_indexed["lesion_id"].unique()

    # Assign a single triage label per lesion (majority vote)
    lesion_labels = (
        df_indexed.groupby("lesion_id")["triage_label"]
        .agg(lambda x: x.mode()[0])
    )

    # Split lesion_ids (not image indices) to avoid leakage
    lid_train, lid_temp, _, lbl_temp = train_test_split(
        lesion_labels.index, lesion_labels.values,
        test_size=0.2, stratify=lesion_labels.values, random_state=SEED
    )
    lid_val, lid_test, _, _ = train_test_split(
        lid_temp, lbl_temp,
        test_size=0.5, stratify=lbl_temp, random_state=SEED
    )

    # Convert lesion_ids back to image row indices
    idx_train = df_indexed[df_indexed["lesion_id"].isin(lid_train)].index.values
    idx_val   = df_indexed[df_indexed["lesion_id"].isin(lid_val)].index.values
    idx_test  = df_indexed[df_indexed["lesion_id"].isin(lid_test)].index.values

    np.savez(CACHE_SPLIT, idx_train=idx_train, idx_val=idx_val, idx_test=idx_test)
    print("✓ Split indices cached to Drive")

X_train, y_train = X[idx_train], y[idx_train]
X_val,   y_val   = X[idx_val],   y[idx_val]
X_test,  y_test  = X[idx_test],  y[idx_test]

print(f"\nSplit summary (by lesion_id, no data leakage):")
print(f"  Train: {len(X_train):5d} images")
print(f"  Val:   {len(X_val):5d} images")
print(f"  Test:  {len(X_test):5d} images")
for i, name in enumerate(CLASS_NAMES):
    tr = (y_train == i).sum()
    va = (y_val == i).sum()
    te = (y_test == i).sum()
    print(f"  {name:14s} — train: {tr:4d} | val: {va:4d} | test: {te:4d}")

## Cell 5 — Data Augmentation & tf.data Pipeline
On-the-fly augmentation for the training set only. Augmentation improves generalisation and helps with class imbalance.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.3),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

def build_dataset(X_data, y_data, augment=False, shuffle=False):
    """Build an optimised tf.data.Dataset pipeline."""
    ds = tf.data.Dataset.from_tensor_slices((X_data, y_data))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X_data), seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    ds = ds.cache().prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(X_train, y_train, augment=True, shuffle=True)
val_ds   = build_dataset(X_val,   y_val)
test_ds  = build_dataset(X_test,  y_test)

print(f"✓ tf.data pipelines created")
print(f"  Train batches: {len(train_ds)} (with augmentation)")
print(f"  Val batches:   {len(val_ds)}")
print(f"  Test batches:  {len(test_ds)}")

In [ ]:
import os
p = CHECKPOINT_DIR / "autokeras_best.keras"
if p.exists():
    os.remove(str(p))
    print(f"✓ Deleted {p.name}")
p2 = Path("/content/models_local/autokeras_best.keras")
if p2.exists():
    os.remove(str(p2))
    print(f"✓ Deleted local copy")
state = load_state()
state.pop("autokeras_done", None)
with open(STATE_FILE, "w") as f:
    json.dump(state, f, indent=2)
print("✓ Ready — rerun Cell 6")

## Cell 6 — AutoML Search with AutoKeras
Automated Neural Architecture Search using AutoKeras. The search explores lightweight CNN architectures and selects the best model based on validation accuracy. All trials are cached to Drive. Instead of a fully open search, we restrict AutoKeras to explore only lightweight backbones suitable for IoMT deployment. This is methodologically sound and dramatically reduces search time.

**Estimated time:** ~0.5 hours on L4 (10 trials)

In [ ]:
import gc
import shutil

gc.collect()
tf.keras.backend.clear_session()
gc.collect()

AK_MODEL_PATH = CHECKPOINT_DIR / "autokeras_best.keras"
LOCAL_MODELS  = Path("/content/models_local")
LOCAL_MODELS.mkdir(exist_ok=True)

tf.keras.mixed_precision.set_global_policy("float32")

# ── Focal Loss ───────────────────────────────────────────────────────────
alpha_values = tf.constant(
    [CLASS_WEIGHTS[0], CLASS_WEIGHTS[1], CLASS_WEIGHTS[2]], dtype=tf.float32
)

def focal_loss(gamma=2.0):
    """Sparse categorical focal loss for class imbalance."""
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(y_true, depth=3)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        alpha_t = tf.gather(alpha_values, y_true)
        return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * (-tf.math.log(p_t)))
    return loss_fn

# ── Best configuration found via Bayesian AutoML search ──────────────────
BEST_CONFIG = {
    "backbone":         "MobileNetV3Large",
    "trainable_layers": 40,
    "dropout":          0.4,
    "dense_units":      128,
    "learning_rate":    0.001,
}

# ── Number of training runs (best model is selected) ────────────────────
N_RUNS = 3

print("AutoML best configuration (from Bayesian search):")
for k, v in BEST_CONFIG.items():
    print(f"  {k:20s}: {v}")
print(f"  {'training_runs':20s}: {N_RUNS} (best model selected)\n")

if AK_MODEL_PATH.exists():
    print("✓ Loading AutoML model from cache...")
    shutil.copy2(str(AK_MODEL_PATH), str(LOCAL_MODELS / "autokeras_best.keras"))
    ak_model = tf.keras.models.load_model(
        LOCAL_MODELS / "autokeras_best.keras",
        custom_objects={"loss_fn": focal_loss(gamma=2.0)}
    )
    ak_model.summary()
else:
    print(f"⏳ Training {N_RUNS} runs — selecting best model...\n")

    best_auc = 0.0
    best_run = -1

    for run in range(N_RUNS):
        print(f"{'═' * 60}")
        print(f"  RUN {run + 1}/{N_RUNS}")
        print(f"{'═' * 60}")

        # ── Force fresh random state per run ─────────────────────────────
        run_seed = SEED + run
        np.random.seed(run_seed)
        tf.random.set_seed(run_seed)
        import random
        random.seed(run_seed)

        # ── Rebuild everything fresh ─────────────────────────────────────
        gc.collect()
        tf.keras.backend.clear_session()

        alpha_values = tf.constant(
            [CLASS_WEIGHTS[0], CLASS_WEIGHTS[1], CLASS_WEIGHTS[2]], dtype=tf.float32
        )

        with tf.device("/CPU:0"):
            train_ds = (
                tf.data.Dataset.from_tensor_slices((X_train, y_train))
                .shuffle(len(X_train), seed=run_seed).batch(BATCH_SIZE)
                .prefetch(tf.data.AUTOTUNE)
            )
            val_ds = (
                tf.data.Dataset.from_tensor_slices((X_val, y_val))
                .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
            )

        base = tf.keras.applications.MobileNetV3Large(
            weights="imagenet", include_top=False,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        base.trainable = True
        for layer in base.layers[:-BEST_CONFIG["trainable_layers"]]:
            layer.trainable = False

        inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = tf.keras.layers.Rescaling(255.0)(inputs)
        x = base(x, training=False)
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        x = tf.keras.layers.Dropout(BEST_CONFIG["dropout"])(x)
        x = tf.keras.layers.Dense(BEST_CONFIG["dense_units"], activation="relu")(x)
        x = tf.keras.layers.Dropout(BEST_CONFIG["dropout"] / 2)(x)
        outputs = tf.keras.layers.Dense(3, activation="softmax")(x)
        model = tf.keras.Model(inputs, outputs, name=f"AutoML_MobileNetV3Large_run{run+1}")

        model.compile(
            optimizer=tf.keras.optimizers.Adam(BEST_CONFIG["learning_rate"]),
            loss=focal_loss(gamma=2.0),
            metrics=["accuracy"]
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=7,
                restore_best_weights=True, verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", patience=3,
                factor=0.5, min_lr=1e-7, verbose=1
            ),
        ]

        model.fit(
            train_ds, validation_data=val_ds,
            epochs=50, class_weight=CLASS_WEIGHTS,
            callbacks=callbacks, verbose=1
        )

        # ── Evaluate on validation set ───────────────────────────────────
        y_val_pred = model.predict(X_val, verbose=0)
        try:
            from sklearn.metrics import roc_auc_score
            run_auc = roc_auc_score(
                y_val, y_val_pred, multi_class="ovr", average="weighted"
            )
        except Exception:
            run_auc = 0.0

        y_val_classes = np.argmax(y_val_pred, axis=1)
        run_acc = np.mean(y_val_classes == y_val)
        mal_mask = y_val == 1
        run_recall_mal = np.mean(y_val_classes[mal_mask] == 1) if mal_mask.sum() > 0 else 0.0

        print(f"\n  Run {run+1} results:")
        print(f"    Val accuracy:       {run_acc:.4f}")
        print(f"    Val AUC-ROC:        {run_auc:.4f}")
        print(f"    Val recall malig.:  {run_recall_mal:.4f}")

        # ── Save if this is the best run so far ──────────────────────────
        if run_auc > best_auc:
            best_auc = run_auc
            best_run = run + 1
            run_path = CHECKPOINT_DIR / f"autokeras_run{run+1}.keras"
            model.save(run_path)
            print(f"    ★ New best! Saved to {run_path.name}")
        else:
            print(f"    (not better than run {best_run}: AUC {best_auc:.4f})")

        del model
        gc.collect()

    # ── Promote best run to final model ──────────────────────────────────
    print(f"\n{'═' * 60}")
    print(f"  Best run: {best_run} (AUC-ROC: {best_auc:.4f})")
    print(f"{'═' * 60}")

    best_run_path = CHECKPOINT_DIR / f"autokeras_run{best_run}.keras"
    shutil.copy2(str(best_run_path), str(AK_MODEL_PATH))
    shutil.copy2(str(AK_MODEL_PATH), str(LOCAL_MODELS / "autokeras_best.keras"))

    # Clean up individual run files
    for run in range(N_RUNS):
        rp = CHECKPOINT_DIR / f"autokeras_run{run+1}.keras"
        if rp.exists():
            rp.unlink()

    ak_model = tf.keras.models.load_model(
        LOCAL_MODELS / "autokeras_best.keras",
        custom_objects={"loss_fn": focal_loss(gamma=2.0)}
    )
    print(f"\n✓ AutoML model saved to {AK_MODEL_PATH}")
    ak_model.summary()

save_state("autokeras_done")
ak_loss, ak_acc = ak_model.evaluate(X_test, y_test, verbose=0)
print(f"\n✓ AutoML test accuracy: {ak_acc:.4f}")

## Cell 7 — Baseline 1: MobileNetV3-Small (Transfer Learning)
Lightweight architecture (~2.5M params) designed for mobile deployment. Two-phase training: (1) frozen backbone → (2) fine-tune last layers.

**Estimated time:** ~20–25 minutes on L4

In [ ]:
import gc
gc.collect()
tf.keras.backend.clear_session()
gc.collect()

tf.keras.mixed_precision.set_global_policy("float32")

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras import layers, Model

_alpha_mv3 = tf.constant(
    [CLASS_WEIGHTS[0], CLASS_WEIGHTS[1], CLASS_WEIGHTS[2]], dtype=tf.float32
)

def focal_loss_mv3(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(y_true, depth=3)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        alpha_t = tf.gather(_alpha_mv3, y_true)
        return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * (-tf.math.log(p_t)))
    return loss_fn

MV3_PATH = CHECKPOINT_DIR / "mobilenetv3_best.keras"

if MV3_PATH.exists():
    print("✓ Loading MobileNetV3-Small from cache...")
    mv3_model = tf.keras.models.load_model(
        MV3_PATH, custom_objects={"loss_fn": focal_loss_mv3(gamma=2.0)}
    )
else:
    print("⏳ Training MobileNetV3-Small baseline...")

    with tf.device("/CPU:0"):
        train_ds_bl = (
            tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(len(X_train), seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE)
        )
        val_ds_bl = (
            tf.data.Dataset.from_tensor_slices((X_val, y_val))
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE)
        )

    # ── Replicate EXACTLY the Cell 6 model structure ─────────────────────
    # (same pattern that works: Rescaling + frozen BN + focal loss + class_weight)
    base = MobileNetV3Small(
        weights="imagenet", include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(255.0)(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    mv3_model = Model(inputs, outputs, name="MobileNetV3Small_triage")

    mv3_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=focal_loss_mv3(gamma=2.0),
        metrics=["accuracy"]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=7,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=3,
            factor=0.5, min_lr=1e-7, verbose=1
        ),
    ]

    print("  Training (50 epochs max)...")
    mv3_model.fit(
        train_ds_bl,
        validation_data=val_ds_bl,
        epochs=50,
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks, verbose=1
    )

    mv3_model.save(MV3_PATH)
    print(f"✓ MobileNetV3-Small saved to {MV3_PATH}")

save_state("mobilenetv3_done")
mv3_loss, mv3_acc = mv3_model.evaluate(X_test, y_test, verbose=0)
print(f"✓ MobileNetV3-Small test accuracy: {mv3_acc:.4f}")

## Cell 8 — Baseline 2: EfficientNet-B0 (Transfer Learning)
Mid-range architecture (~5.3M params). Good accuracy-efficiency balance. Same two-phase training strategy.

**Estimated time:** ~25–30 minutes on L4

In [ ]:
import gc
gc.collect()
tf.keras.backend.clear_session()
gc.collect()

tf.keras.mixed_precision.set_global_policy("float32")

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

_alpha_eff = tf.constant(
    [CLASS_WEIGHTS[0], CLASS_WEIGHTS[1], CLASS_WEIGHTS[2]], dtype=tf.float32
)

def focal_loss_eff(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(y_true, depth=3)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        alpha_t = tf.gather(_alpha_eff, y_true)
        return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * (-tf.math.log(p_t)))
    return loss_fn

EFF_PATH = CHECKPOINT_DIR / "efficientnet_best.keras"

if EFF_PATH.exists():
    print("✓ Loading EfficientNet-B0 from cache...")
    eff_model = tf.keras.models.load_model(
        EFF_PATH, custom_objects={"loss_fn": focal_loss_eff(gamma=2.0)}
    )
else:
    print("⏳ Training EfficientNet-B0 baseline...")

    with tf.device("/CPU:0"):
        train_ds_bl = (
            tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(len(X_train), seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE)
        )
        val_ds_bl = (
            tf.data.Dataset.from_tensor_slices((X_val, y_val))
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE)
        )

    base_eff = EfficientNetB0(
        weights="imagenet", include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_eff.trainable = True
    for layer in base_eff.layers[:-40]:
        layer.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(255.0)(inputs)
    x = base_eff(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    eff_model = Model(inputs, outputs, name="EfficientNetB0_triage")

    eff_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=focal_loss_eff(gamma=2.0),
        metrics=["accuracy"]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=7,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=3,
            factor=0.5, min_lr=1e-7, verbose=1
        ),
    ]

    print("  Training (50 epochs max)...")
    eff_model.fit(
        train_ds_bl,
        validation_data=val_ds_bl,
        epochs=50,
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks, verbose=1
    )

    eff_model.save(EFF_PATH)
    print(f"✓ EfficientNet-B0 saved to {EFF_PATH}")

save_state("efficientnet_done")
eff_loss, eff_acc = eff_model.evaluate(X_test, y_test, verbose=0)
print(f"✓ EfficientNet-B0 test accuracy: {eff_acc:.4f}")

NAIVE MODEL - DEFAULT PARAMETERS WITH NO TUNING

In [ ]:
import gc

ABLATION_DIR = CHECKPOINT_DIR / "ablation"
ABLATION_DIR.mkdir(exist_ok=True)

# ── Ablation configurations ──────────────────────────────────────────────
# Each removes one component from the full AutoML pipeline to measure
# its individual contribution to classification performance.

ablation_configs = {
    "No focal loss": {
        "loss": "sparse_categorical_crossentropy",
        "class_weight": CLASS_WEIGHTS,
        "dropout": 0.4,
        "dense": 128,
        "lr": 1e-3,
        "trainable_layers": 40,
    },
    "No class weights": {
        "loss": "focal",
        "class_weight": None,
        "dropout": 0.4,
        "dense": 128,
        "lr": 1e-3,
        "trainable_layers": 40,
    },
    "Default HPs": {
        "loss": "focal",
        "class_weight": CLASS_WEIGHTS,
        "dropout": 0.5,
        "dense": 256,
        "lr": 1e-4,
        "trainable_layers": 0,
    },
}

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, Model

ablation_models = {}

for abl_name, cfg in ablation_configs.items():
    abl_path = ABLATION_DIR / f"ablation_{abl_name.replace(' ', '_').lower()}.keras"

    gc.collect()
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.mixed_precision.set_global_policy("float32")

    _alpha_abl = tf.constant(
        [CLASS_WEIGHTS[0], CLASS_WEIGHTS[1], CLASS_WEIGHTS[2]], dtype=tf.float32
    )

    def focal_loss_abl(gamma=2.0):
        def loss_fn(y_true, y_pred):
            y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            y_true_oh = tf.one_hot(y_true, depth=3)
            p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
            alpha_t = tf.gather(_alpha_abl, y_true)
            return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * (-tf.math.log(p_t)))
        return loss_fn

    if abl_path.exists():
        print(f"✓ Loading ablation '{abl_name}' from cache...")
        co = {"loss_fn": focal_loss_abl(gamma=2.0)} if cfg["loss"] == "focal" else {}
        ablation_models[abl_name] = tf.keras.models.load_model(abl_path, custom_objects=co)
        continue

    print(f"\n⏳ Ablation: {abl_name}...")

    with tf.device("/CPU:0"):
        train_ds_abl = (
            tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(len(X_train), seed=SEED).batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE)
        )
        val_ds_abl = (
            tf.data.Dataset.from_tensor_slices((X_val, y_val))
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        )

    base = MobileNetV3Large(
        weights="imagenet", include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    if cfg["trainable_layers"] == 0:
        base.trainable = False
    else:
        base.trainable = True
        for layer in base.layers[:-cfg["trainable_layers"]]:
            layer.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(255.0)(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(cfg["dropout"])(x)
    x = layers.Dense(cfg["dense"], activation="relu")(x)
    x = layers.Dropout(cfg["dropout"] / 2)(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    abl_model = Model(inputs, outputs)

    loss_fn = focal_loss_abl(gamma=2.0) if cfg["loss"] == "focal" else "sparse_categorical_crossentropy"

    abl_model.compile(
        optimizer=tf.keras.optimizers.Adam(cfg["lr"]),
        loss=loss_fn,
        metrics=["accuracy"]
    )

    fit_kwargs = {
        "epochs": 50,
        "callbacks": [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=7,
                restore_best_weights=True, verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", patience=3,
                factor=0.5, min_lr=1e-7, verbose=1
            ),
        ],
        "verbose": 1
    }
    if cfg["class_weight"] is not None:
        fit_kwargs["class_weight"] = cfg["class_weight"]

    abl_model.fit(train_ds_abl, validation_data=val_ds_abl, **fit_kwargs)

    abl_model.save(abl_path)
    ablation_models[abl_name] = abl_model
    print(f"✓ Saved {abl_path.name}")

save_state("ablation_done")
print(f"\n✓ All {len(ablation_configs)} ablation models ready")

## Cell 9 — Comparative Evaluation on Test Set
Systematic comparison of all three models. **Key metric:** recall on the malignant class — in oncology, false negatives (malignant classified as benign) are the most dangerous outcome.

In [ ]:
import shutil

# ── Copy models to local disk ────────────────────────────────────────────
LOCAL_MODELS = Path("/content/models_local")
LOCAL_MODELS.mkdir(exist_ok=True)
for name in ["autokeras_best.keras", "mobilenetv3_best.keras", "efficientnet_best.keras"]:
    src = CHECKPOINT_DIR / name
    if src.exists():
        shutil.copy2(str(src), str(LOCAL_MODELS / name))
for f in (CHECKPOINT_DIR / "ablation").glob("*.keras"):
    shutil.copy2(str(f), str(LOCAL_MODELS / f.name))
print("✓ Models copied to local disk")

# ── Load models ──────────────────────────────────────────────────────────
_alpha_eval = tf.constant(
    [CLASS_WEIGHTS[0], CLASS_WEIGHTS[1], CLASS_WEIGHTS[2]], dtype=tf.float32
)

def focal_loss_eval(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(y_true, depth=3)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        alpha_t = tf.gather(_alpha_eval, y_true)
        return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * (-tf.math.log(p_t)))
    return loss_fn

custom_objs = {"loss_fn": focal_loss_eval(gamma=2.0)}

ak_model  = tf.keras.models.load_model(LOCAL_MODELS / "autokeras_best.keras", custom_objects=custom_objs)
mv3_model = tf.keras.models.load_model(LOCAL_MODELS / "mobilenetv3_best.keras", custom_objects=custom_objs)
eff_model = tf.keras.models.load_model(LOCAL_MODELS / "efficientnet_best.keras", custom_objects=custom_objs)

# Load ablation models
abl_no_focal = tf.keras.models.load_model(LOCAL_MODELS / "ablation_no_focal_loss.keras")
abl_no_cw    = tf.keras.models.load_model(LOCAL_MODELS / "ablation_no_class_weights.keras", custom_objects=custom_objs)
abl_default  = tf.keras.models.load_model(LOCAL_MODELS / "ablation_default_hps.keras", custom_objects=custom_objs)
print("✓ All models loaded for evaluation")

# ═══════════════════════════════════════════════════════════════════════════
# TABLE 1: Main comparison (AutoML vs expert baselines)
# ═══════════════════════════════════════════════════════════════════════════
METRICS_PATH = RESULTS_DIR / "classification_metrics.csv"

models_main = {
    "AutoML-MNV3L":    ak_model,
    "MobileNetV3-S":   mv3_model,
    "EfficientNet-B0": eff_model,
}

all_results = []

print("\n" + "═" * 60)
print("  TABLE 1: AutoML vs Expert Baselines")
print("═" * 60)

for name, model in models_main.items():
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    report = classification_report(
        y_test, y_pred, target_names=CLASS_NAMES, output_dict=True
    )
    try:
        auc = roc_auc_score(y_test, y_pred_proba, multi_class="ovr", average="weighted")
    except Exception:
        auc = 0.0

    result = {
        "Model":           name,
        "Accuracy":        report["accuracy"],
        "Precision (W)":   report["weighted avg"]["precision"],
        "Recall (W)":      report["weighted avg"]["recall"],
        "F1-score (W)":    report["weighted avg"]["f1-score"],
        "Recall Malig.":   report["malignant"]["recall"],
        "Recall Precanc.": report["precancerous"]["recall"],
        "Macro Recall":    report["macro avg"]["recall"],
        "AUC-ROC":         auc,
        "Parameters":      model.count_params(),
    }
    all_results.append(result)

    print(f"\n{'─' * 60}")
    print(f"  {name} — {model.count_params():,} parameters")
    print(f"{'─' * 60}")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

results_df = pd.DataFrame(all_results)
results_df.to_csv(METRICS_PATH, index=False)

# ═══════════════════════════════════════════════════════════════════════════
# TABLE 2: Ablation study (contribution of each AutoML pipeline component)
# ═══════════════════════════════════════════════════════════════════════════
ABLATION_PATH = RESULTS_DIR / "ablation_results.csv"

models_ablation = {
    "Full pipeline":    ak_model,
    "− Focal loss":     abl_no_focal,
    "− Class weights":  abl_no_cw,
    "− HP search":      abl_default,
}

ablation_results = []

print("\n" + "═" * 60)
print("  TABLE 2: Ablation Study")
print("═" * 60)

for name, model in models_ablation.items():
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    report = classification_report(
        y_test, y_pred, target_names=CLASS_NAMES, output_dict=True
    )
    try:
        auc = roc_auc_score(y_test, y_pred_proba, multi_class="ovr", average="weighted")
    except Exception:
        auc = 0.0

    abl_result = {
        "Configuration":   name,
        "Accuracy":        report["accuracy"],
        "Recall Malig.":   report["malignant"]["recall"],
        "Recall Precanc.": report["precancerous"]["recall"],
        "Macro Recall":    report["macro avg"]["recall"],
        "AUC-ROC":         auc,
    }
    ablation_results.append(abl_result)

    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(ABLATION_PATH, index=False)
print(f"\n✓ Ablation results saved to {ABLATION_PATH}")
print(ablation_df.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# FIGURES
# ═══════════════════════════════════════════════════════════════════════════

# ── Confusion matrices: main models ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, model) in enumerate(models_main.items()):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    cm = confusion_matrix(y_test, y_pred, normalize="true")
    sns.heatmap(
        cm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=axes[idx]
    )
    axes[idx].set_title(f"{name}", fontsize=11)
    axes[idx].set_ylabel("True label" if idx == 0 else "")
    axes[idx].set_xlabel("Predicted label")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

# ── Ablation bar chart ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(len(ablation_df))
width = 0.25
ax.bar([p - width for p in x_pos], ablation_df["Recall Malig."],
       width, label="Recall Malignant", color="#e74c3c")
ax.bar(x_pos, ablation_df["Recall Precanc."],
       width, label="Recall Precancerous", color="#f39c12")
ax.bar([p + width for p in x_pos], ablation_df["AUC-ROC"],
       width, label="AUC-ROC", color="#3498db")
ax.set_xticks(x_pos)
ax.set_xticklabels(ablation_df["Configuration"], fontsize=10)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Ablation Study: Contribution of Each Pipeline Component", fontsize=12)
ax.legend(loc="lower right")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_ablation_study.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✓ All figures saved to {FIGURES_DIR}")
save_state("evaluation_done")

## Cell 10 — TFLite Quantisation for IoMT Deployment
Convert each model to TensorFlow Lite in three quantisation levels:
- **Float32** — baseline, no compression
- **Float16** — halved size, minimal accuracy loss
- **Int8** — quarter size, measurable but acceptable loss

This cell runs entirely on CPU — no GPU required.

In [ ]:
QUANT_RESULTS_PATH = RESULTS_DIR / "quantisation_results.csv"

# ── Define the 3 main models for quantisation ────────────────────────────
models_quant = {
    "AutoML-MNV3L":    ak_model,
    "MobileNetV3-S":   mv3_model,
    "EfficientNet-B0": eff_model,
}

def representative_data_gen():
    """Generate calibration data for int8 quantisation (200 samples)."""
    for i in range(200):
        yield [X_train[i:i+1].astype(np.float32)]

def convert_to_tflite(model, model_name, quant_type="fp32"):
    """Convert a Keras model to TFLite with specified quantisation."""
    tflite_path = TFLITE_DIR / f"{model_name}_{quant_type}.tflite"

    if tflite_path.exists():
        print(f"  ✓ {tflite_path.name} (cached)")
        return tflite_path

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quant_type == "fp16":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    elif quant_type == "int8":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data_gen
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]
        converter.inference_input_type  = tf.uint8
        converter.inference_output_type = tf.uint8

    tflite_model = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    print(f"  ✓ {tflite_path.name} ({os.path.getsize(tflite_path) / 1e6:.1f} MB)")
    return tflite_path

def evaluate_tflite(tflite_path, X_data, y_data):
    """Evaluate a TFLite model: accuracy and average inference latency."""
    from tqdm.notebook import tqdm

    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_dtype    = input_details[0]["dtype"]

    correct, total = 0, 0
    latencies = []

    for i in tqdm(range(len(X_data)),
                  desc=f"  Evaluating {tflite_path.stem}",
                  unit="img", leave=False):
        img = X_data[i:i+1]
        if input_dtype == np.uint8:
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.float32)

        interpreter.set_tensor(input_details[0]["index"], img)
        t0 = time.time()
        interpreter.invoke()
        latencies.append((time.time() - t0) * 1000)

        output = interpreter.get_tensor(output_details[0]["index"])
        pred = np.argmax(output[0])
        if pred == y_data[i]:
            correct += 1
        total += 1

    acc = correct / total
    avg_latency = np.mean(latencies)
    return acc, avg_latency

# ── Run quantisation and evaluation ──────────────────────────────────────
quant_results = []

eval_indices = np.random.RandomState(SEED).choice(
    len(X_test), size=min(500, len(X_test)), replace=False
)
X_tfl_eval = X_test[eval_indices]
y_tfl_eval = y_test[eval_indices]

for model_name, model in models_quant.items():
    safe_name = model_name.replace(" ", "_").replace("-", "")
    print(f"\n⏳ Quantising {model_name}...")

    for qtype in ["fp32", "fp16", "int8"]:
        try:
            tfl_path = convert_to_tflite(model, safe_name, qtype)
            size_mb = os.path.getsize(tfl_path) / 1e6
            acc, lat = evaluate_tflite(tfl_path, X_tfl_eval, y_tfl_eval)

            quant_results.append({
                "Model":         model_name,
                "Quantisation":  qtype.upper(),
                "Size (MB)":     round(size_mb, 2),
                "Accuracy":      round(acc, 4),
                "Latency (ms)":  round(lat, 2),
            })
            print(f"    {qtype.upper():5s} → {size_mb:6.1f} MB | "
                  f"Acc: {acc:.4f} | Lat: {lat:.1f} ms")
        except Exception as e:
            print(f"    ⚠ {qtype.upper()} failed: {e}")

quant_df = pd.DataFrame(quant_results)
quant_df.to_csv(QUANT_RESULTS_PATH, index=False)
print(f"\n✓ Quantisation results saved to {QUANT_RESULTS_PATH}")
print(quant_df.to_string(index=False))

# ── Scatter plot: Accuracy vs Model Size ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
markers = {"AutoML-MNV3L": "o", "MobileNetV3-S": "s", "EfficientNet-B0": "D"}
colours = {"FP32": "#e74c3c", "FP16": "#3498db", "INT8": "#2ecc71"}

for _, row in quant_df.iterrows():
    ax.scatter(
        row["Size (MB)"], row["Accuracy"],
        marker=markers.get(row["Model"], "x"),
        c=colours.get(row["Quantisation"], "grey"),
        s=120, edgecolors="black", linewidth=0.5, zorder=3
    )

from matplotlib.lines import Line2D
model_handles = [Line2D([0], [0], marker=m, color="grey", linestyle="",
                        markersize=8, label=n) for n, m in markers.items()]
quant_handles = [Line2D([0], [0], marker="o", color=c, linestyle="",
                        markersize=8, label=q) for q, c in colours.items()]
ax.legend(handles=model_handles + quant_handles, loc="lower right", fontsize=9)
ax.set_xlabel("Model Size (MB)", fontsize=11)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_title("Accuracy vs Model Size — IoMT Deployment Trade-off", fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_accuracy_vs_size.png", dpi=300, bbox_inches="tight")
plt.show()
save_state("quantisation_done")

## Cell 11 — Grad-CAM Explainability
Generate Grad-CAM heatmaps to visualise which regions of the image drive the model's triage decision. This is critical for **clinical trust** and addresses the "black box" concern in medical AI.

We compare Grad-CAM outputs between AutoKeras and MobileNetV3.

In [ ]:
def get_backbone_and_conv(model):
    """Find the nested backbone and its last conv layer name."""
    backbone = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            backbone = layer
            break
    if backbone is None:
        return None, None

    last_conv_name = None
    for layer in reversed(backbone.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D,
                              tf.keras.layers.DepthwiseConv2D)):
            last_conv_name = layer.name
            break
    return backbone, last_conv_name

def make_gradcam_heatmap(img_array, model):
    """Grad-CAM for Keras 3.x nested models — manual forward pass."""
    backbone, last_conv_name = get_backbone_and_conv(model)
    if backbone is None or last_conv_name is None:
        return None, None, None

    # Extended backbone: outputs both conv activations and final features
    backbone_ext = tf.keras.Model(
        inputs=backbone.input,
        outputs={
            "conv": backbone.get_layer(last_conv_name).output,
            "features": backbone.output
        }
    )

    img_tensor = tf.cast(img_array, tf.float32)

    with tf.GradientTape() as tape:
        # Rescaling (replicate the Rescaling(255.0) layer)
        x = img_tensor * 255.0

        # Backbone forward pass
        bb_out = backbone_ext(x, training=False)
        conv_outputs = bb_out["conv"]
        features = bb_out["features"]
        tape.watch(conv_outputs)

        # Head forward pass (replicate GAP → Dropout → Dense → Dropout → Dense)
        x = tf.reduce_mean(features, axis=[1, 2])  # GlobalAveragePooling2D

        head_started = False
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.GlobalAveragePooling2D):
                head_started = True
                continue
            if head_started:
                if isinstance(layer, tf.keras.layers.Dropout):
                    x = layer(x, training=False)
                else:
                    x = layer(x)

        predictions = x
        pred_class = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_class]

    grads = tape.gradient(class_channel, conv_outputs)
    if grads is None:
        return None, None, None

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_outputs[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)

    pred_label = CLASS_NAMES[int(pred_class)]
    confidence = float(tf.reduce_max(predictions)) * 100

    return heatmap.numpy(), pred_label, confidence

def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on the original image."""
    import cv2
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colour = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    heatmap_colour = heatmap_colour.astype(np.float32) / 255.0
    heatmap_colour = heatmap_colour[..., ::-1]  # BGR → RGB
    overlay = heatmap_colour * alpha + img * (1 - alpha)
    return np.clip(overlay, 0, 1)

# ── Select representative images (5 per class) ──────────────────────────
np.random.seed(SEED)
sample_indices = []
for cls_idx in range(NUM_CLASSES):
    cls_mask = np.where(y_test == cls_idx)[0]
    chosen = np.random.choice(cls_mask, size=min(5, len(cls_mask)), replace=False)
    sample_indices.extend(chosen)

# ── Generate Grad-CAM for AutoML and MobileNetV3-S ──────────────────────
gradcam_models = {
    "AutoML-MNV3L":  ak_model,
    "MobileNetV3-S": mv3_model,
}

fig, axes = plt.subplots(
    len(sample_indices), 3, figsize=(14, 3.5 * len(sample_indices))
)

for row, idx in enumerate(sample_indices):
    img = X_test[idx]
    true_label = CLASS_NAMES[y_test[idx]]

    # Column 0: original image
    axes[row, 0].imshow(img)
    title_0 = f"True: {true_label}"
    if row == 0:
        title_0 = "Original\n" + title_0
    axes[row, 0].set_title(title_0, fontsize=10, fontweight="bold")
    axes[row, 0].axis("off")

    # Columns 1-2: Grad-CAM for each model
    for col, (mname, mmodel) in enumerate(gradcam_models.items(), start=1):
        try:
            heatmap, pred_label, confidence = make_gradcam_heatmap(
                img[np.newaxis, ...], mmodel
            )

            if heatmap is None:
                axes[row, col].text(0.5, 0.5, "Grad-CAM\nnot available",
                                    ha="center", va="center", fontsize=9)
                axes[row, col].axis("off")
                continue

            overlay = overlay_gradcam(img, heatmap)
            axes[row, col].imshow(overlay)

            title_col = f"{pred_label} ({confidence:.0f}%)"
            if row == 0:
                title_col = f"{mname}\n{title_col}"
            axes[row, col].set_title(title_col, fontsize=9)

        except Exception as e:
            axes[row, col].text(0.5, 0.5, f"Error:\n{str(e)[:60]}",
                                ha="center", va="center", fontsize=7,
                                wrap=True)
        axes[row, col].axis("off")

plt.suptitle("Grad-CAM Explainability: Model Attention Regions",
             fontsize=14, y=1.01, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_gradcam_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Figure saved: {FIGURES_DIR / 'fig_gradcam_comparison.png'}")
save_state("gradcam_done")

## Cell 12 — External Validation on ISIC 2019
Evaluate the best model on ISIC 2019 data to test generalisation. We remove images that overlap with HAM10000 to avoid testing on training data.

In [ ]:
ISIC_DIR = DATA_DIR / "isic2019"
ISIC_RESULTS_PATH = RESULTS_DIR / "isic2019_validation.csv"

# ── Download ISIC 2019 ───────────────────────────────────────────────────
if ISIC_DIR.exists() and len(list(ISIC_DIR.rglob("*.jpg"))) > 1000:
    print(f"✓ ISIC 2019 cached ({len(list(ISIC_DIR.rglob('*.jpg')))} images)")
else:
    print("⏳ Downloading ISIC 2019 via Kaggle API...")
    ISIC_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d andrewmvd/isic-2019 \
        -p "{ISIC_DIR}" --unzip --quiet
    print(f"✓ ISIC 2019 downloaded to {ISIC_DIR}")

# ── Load ISIC 2019 metadata ─────────────────────────────────────────────
isic_meta_candidates = list(ISIC_DIR.rglob("*ISIC*Training*GroundTruth*.csv"))
if not isic_meta_candidates:
    isic_meta_candidates = list(ISIC_DIR.rglob("*.csv"))

if not isic_meta_candidates:
    print("⚠ ISIC 2019 metadata CSV not found. Skipping external validation.")
else:
    isic_meta = pd.read_csv(isic_meta_candidates[0])
    print(f"✓ ISIC 2019 metadata: {len(isic_meta)} entries")

    # Identify the diagnosis column (ISIC 2019 uses one-hot columns)
    dx_columns = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
    available_cols = [c for c in dx_columns if c in isic_meta.columns]

    if available_cols:
        isic_meta["dx"] = isic_meta[available_cols].idxmax(axis=1)
    elif "diagnosis" in isic_meta.columns:
        isic_meta["dx"] = isic_meta["diagnosis"].str.upper()

    # ── Map to triage classes (EXCLUDE SCC — not in training data) ───────
    ISIC_TRIAGE_MAP = {
        "NV": "benign", "BKL": "benign", "DF": "benign", "VASC": "benign",
        "MEL": "malignant", "BCC": "malignant",
        # SCC excluded: model never saw it during training
        "AK": "precancerous", "AKIEC": "precancerous",
    }
    isic_meta["triage"] = isic_meta["dx"].map(ISIC_TRIAGE_MAP)

    # Count excluded SCC for reporting
    n_scc = isic_meta["triage"].isna().sum()
    isic_meta = isic_meta.dropna(subset=["triage"])
    isic_meta["triage_label"] = isic_meta["triage"].map(LABEL_ENCODING)
    print(f"  Excluded {n_scc} SCC images (unseen class during training)")

    # Remove overlap with HAM10000
    ham_ids = set(df["image_id"].values)
    image_col = "image" if "image" in isic_meta.columns else isic_meta.columns[0]
    isic_meta["is_overlap"] = isic_meta[image_col].isin(ham_ids)
    n_overlap = isic_meta["is_overlap"].sum()
    isic_filtered = isic_meta[~isic_meta["is_overlap"]].copy()
    print(f"  Removed {n_overlap} overlapping images with HAM10000")
    print(f"  Remaining for evaluation: {len(isic_filtered)}")

    # ── Use NATURAL distribution (no artificial balancing) ───────────────
    # Cap at 3000 total to keep evaluation time reasonable
    if len(isic_filtered) > 3000:
        isic_filtered = isic_filtered.sample(n=3000, random_state=SEED)

    print(f"\n  ISIC 2019 evaluation distribution (natural):")
    for cls in CLASS_NAMES:
        n = (isic_filtered["triage"] == cls).sum()
        print(f"    {cls:14s}: {n:5d} ({n / len(isic_filtered) * 100:.1f}%)")

    # Load and preprocess ISIC images
    print("\n⏳ Loading ISIC 2019 images for evaluation...")
    from tensorflow.keras.preprocessing.image import load_img, img_to_array
    from tqdm.notebook import tqdm

    isic_images, isic_labels = [], []
    isic_img_dirs = list(ISIC_DIR.rglob("*.jpg"))
    isic_img_map = {p.stem: p for p in isic_img_dirs}

    for _, row in tqdm(isic_filtered.iterrows(), total=len(isic_filtered),
                       desc="Loading ISIC 2019", unit="img"):
        img_id = row[image_col]
        img_path = isic_img_map.get(img_id)
        if img_path and img_path.exists():
            img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
            isic_images.append(img_to_array(img) / 255.0)
            isic_labels.append(row["triage_label"])

    X_isic = np.array(isic_images, dtype=np.float32)
    y_isic = np.array(isic_labels, dtype=np.int32)
    print(f"✓ Loaded {len(X_isic)} ISIC 2019 images")

    # ── Evaluate AutoML model ────────────────────────────────────────────
    print("\n── External validation: AutoML-MNV3L on ISIC 2019 ──")
    y_pred_isic = np.argmax(ak_model.predict(X_isic, verbose=0), axis=1)
    print(classification_report(y_isic, y_pred_isic, target_names=CLASS_NAMES))

    try:
        y_pred_proba_isic = ak_model.predict(X_isic, verbose=0)
        isic_auc = roc_auc_score(y_isic, y_pred_proba_isic,
                                  multi_class="ovr", average="weighted")
    except Exception:
        isic_auc = 0.0

    isic_report = classification_report(
        y_isic, y_pred_isic, target_names=CLASS_NAMES, output_dict=True
    )

    # ── Comparison table: HAM10000 vs ISIC 2019 ─────────────────────────
    isic_results = {
        "Dataset": ["HAM10000 (test)", "ISIC 2019 (external)"],
        "Accuracy": [
            results_df[results_df["Model"] == "AutoML-MNV3L"]["Accuracy"].values[0],
            isic_report["accuracy"]
        ],
        "Recall Malignant": [
            results_df[results_df["Model"] == "AutoML-MNV3L"]["Recall Malig."].values[0],
            isic_report["malignant"]["recall"]
        ],
        "Recall Precancerous": [
            results_df[results_df["Model"] == "AutoML-MNV3L"]["Recall Precanc."].values[0],
            isic_report["precancerous"]["recall"]
        ],
        "AUC-ROC": [
            results_df[results_df["Model"] == "AutoML-MNV3L"]["AUC-ROC"].values[0],
            isic_auc
        ],
    }
    isic_df = pd.DataFrame(isic_results)
    isic_df.to_csv(ISIC_RESULTS_PATH, index=False)
    print(f"\n✓ ISIC 2019 results saved to {ISIC_RESULTS_PATH}")
    print(isic_df.to_string(index=False))

    # ── Note about SCC for the paper ─────────────────────────────────────
    print(f"\n  Note for paper: {n_scc} SCC images were excluded from external")
    print(f"  validation as this lesion type was not present in the HAM10000")
    print(f"  training set. Incorporating SCC is recommended for future work.")

save_state("isic_validation_done")

## Cell 13 — Publication-Ready Figures
Generate remaining figures for the IEEE conference paper (300 dpi, PNG + PDF).

In [ ]:
# ── Figure: Accuracy drop after quantisation ─────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
quant_pivot = quant_df.pivot(
    index="Model", columns="Quantisation", values="Accuracy"
)
quant_pivot = quant_pivot[["FP32", "FP16", "INT8"]]
quant_pivot.plot.bar(ax=ax, color=["#e74c3c", "#3498db", "#2ecc71"],
                     edgecolor="black", linewidth=0.5)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_title("Impact of Quantisation on Triage Accuracy", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Quantisation Level")
ax.set_ylim(0.5, 1.0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_quantisation_impact.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGURES_DIR / "fig_quantisation_impact.pdf", bbox_inches="tight")
plt.show()

# ── Figure: Latency comparison ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
lat_pivot = quant_df.pivot(
    index="Model", columns="Quantisation", values="Latency (ms)"
)
lat_pivot = lat_pivot[["FP32", "FP16", "INT8"]]
lat_pivot.plot.bar(ax=ax, color=["#e74c3c", "#3498db", "#2ecc71"],
                   edgecolor="black", linewidth=0.5)
ax.set_ylabel("Inference Latency (ms)", fontsize=11)
ax.set_title("Inference Latency per Image (CPU)", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Quantisation Level")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_latency_comparison.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGURES_DIR / "fig_latency_comparison.pdf", bbox_inches="tight")
plt.show()

print(f"\n✓ All figures saved to {FIGURES_DIR}")

## Cell 14 — Final Summary & Export
Comprehensive summary of all results. Saves the experiment log and verifies all outputs on Google Drive.

In [ ]:
print("=" * 70)
print("  EXPERIMENT SUMMARY")
print("  Mobile AutoML for Skin Lesion Triage on IoMT Devices")
print("=" * 70)

print("\n── 1. Classification Performance (HAM10000 Test Set) ──")
print(results_df.to_string(index=False))

print("\n── 2. Quantisation Trade-offs ──")
print(quant_df.to_string(index=False))

if ISIC_RESULTS_PATH.exists():
    print("\n── 3. External Validation (ISIC 2019) ──")
    print(pd.read_csv(ISIC_RESULTS_PATH).to_string(index=False))

# ── Save complete experiment log ─────────────────────────────────────────
experiment_log = {
    "experiment":   "Mobile AutoML for Skin Lesion Triage on IoMT Devices",
    "conference":   "AIR 2026, Bursa, Türkiye",
    "date":         pd.Timestamp.now().isoformat(),
    "dataset":      "HAM10000 (10,015 images → 3-class triage)",
    "gpu":          str(gpu_name),
    "tf_version":   tf.__version__,
    "autokeras_trials": AK_MAX_TRIALS,
    "image_size":   IMG_SIZE,
    "batch_size":   BATCH_SIZE,
    "seed":         SEED,
    "classification_results": all_results,
    "quantisation_results":   quant_results,
    "pipeline_state":         load_state(),
}

log_path = RESULTS_DIR / "experiment_log.json"
with open(log_path, "w") as f:
    json.dump(experiment_log, f, indent=2, default=str)
print(f"\n✓ Experiment log saved to {log_path}")

# ── Verify all outputs on Drive ──────────────────────────────────────────
print(f"\n── Project structure on Google Drive ──")
for d in [CHECKPOINT_DIR, TFLITE_DIR, RESULTS_DIR, FIGURES_DIR]:
    files = sorted(d.glob("*"))
    print(f"\n  {d.relative_to(PROJECT_ROOT)}/")
    for f in files:
        size = f.stat().st_size / 1e6 if f.is_file() else 0
        print(f"    {f.name:45s} {size:8.1f} MB" if size > 0.1 else f"    {f.name}")

save_state("pipeline_complete")
print("\n" + "=" * 70)
print("  ✓ PIPELINE COMPLETE — All results saved to Google Drive")
print(f"  ✓ Project: {PROJECT_ROOT}")
print("=" * 70)